# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Extract Creator Open IDs

In [4]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
list_all_creators = df_creators_list['username'].tolist()

In [5]:
# Load or initialize the manifest (tracks which handles are already found)
if Path(MANIFEST_CSV).exists():
    manifest = pd.read_csv(MANIFEST_CSV)
else:
    manifest = pd.DataFrame({"handle": list_all_creators, "found": False})

# Load or initialize consolidated creator data pulled so far
if Path(CONSOLIDATED_CSV).exists():
    df_creators = pd.read_csv(CONSOLIDATED_CSV)
else:
    df_creators = pd.DataFrame()

# recheck manifest in case of failed runs
manifest["found"] = manifest["handle"].isin(set(df_creators['username']))
manifest.to_csv(MANIFEST_CSV, index=False)

# check handles to find
handles_to_find = manifest.loc[~manifest["found"], "handle"].tolist()
print(f"{len(handles_to_find)} handles left to find out of {len(manifest)} total.\n")

20879 handles left to find out of 21598 total.



In [6]:
manifest['found'].sum()

np.int64(719)

In [7]:
# finish top 2k handles first
top_n = 300
handles_to_find = handles_to_find[:top_n]

## First-pass: Chunk size = 5

In [8]:
# Phase 1: chunk_size=5, single pass through everything not yet found
still_not_found, found_usernames, df_creators = run_pass(handles_to_find, chunk_size=5, df_creators=df_creators)

manifest["found"] = manifest["handle"].isin(found_usernames)
manifest.to_csv(MANIFEST_CSV, index=False)
print(f"\nPhase 1 done. {len(still_not_found)} handles still not found.\n")

[chunk_size=5] 1/60: searching ['ginoroqueiv', 'agapearliyshop', 'rosmar.acaiberry.legit', 'khatemekhate003', 'missmeljc']
  ⚠️  Still rate-limited after 3 attempts — giving up for now.
[chunk_size=5] 2/60: searching ['jmpiamonte4', 'nurse.aiz', 'elmaasagra', 'jhonlouiewasawas95', 'yellowcandyace2']
[chunk_size=5] 3/60: searching ['miimasaur', 'skincarechuchay', 'shopaholic_sister', 'thisisliezel0', 'shoppersvirtual']
  ⚠️  Still rate-limited after 3 attempts — giving up for now.
[chunk_size=5] 4/60: searching ['recosbynesa', 'findingehl22', 'jhoanagrasya0614', 'lia.xx2x', 'itsmekitkat_01']
  ⚠️  Still rate-limited after 3 attempts — giving up for now.
[chunk_size=5] 5/60: searching ['lexxofficial1', 'cindyloraaa', 'blaisse', 'reiseyeee_06', 'lhenies']
[chunk_size=5] 6/60: searching ['flfychm', 'iladongspotted24', 'lovely.anne21', 'youbeauty_hub', 'mariellaaguinaldo']
[chunk_size=5] 7/60: searching ['danibarrettop', 'walalangto313', 'trendeezone', 'assortedproductshopping', 'annecorpz'

## Second-pass: Chunk size = 1

In [9]:
# Phase 2: chunk_size=1, only for leftovers from phase 1
if still_not_found:
    still_not_found, found_usernames, df_creators = run_pass(still_not_found, chunk_size=1, df_creators=df_creators)

    found_usernames = set(df_creators["username"])
    manifest["found"] = manifest["handle"].isin(found_usernames)
    manifest.to_csv(MANIFEST_CSV, index=False)

print(f"\nDone. {int(manifest['found'].sum())} / {len(manifest)} handles found.")

[chunk_size=1] 1/189: searching ['ginoroqueiv']
[chunk_size=1] 2/189: searching ['agapearliyshop']
  ⚠️  Still rate-limited after 3 attempts — giving up for now.
[chunk_size=1] 3/189: searching ['rosmar.acaiberry.legit']
[chunk_size=1] 4/189: searching ['khatemekhate003']
[chunk_size=1] 5/189: searching ['missmeljc']
  ⚠️  Still rate-limited after 3 attempts — giving up for now.
[chunk_size=1] 6/189: searching ['jmpiamonte4']
[chunk_size=1] 7/189: searching ['miimasaur']
[chunk_size=1] 8/189: searching ['skincarechuchay']
  ⚠️  Still rate-limited after 3 attempts — giving up for now.
[chunk_size=1] 9/189: searching ['shopaholic_sister']
[chunk_size=1] 10/189: searching ['thisisliezel0']
[chunk_size=1] 11/189: searching ['shoppersvirtual']
  ⚠️  Still rate-limited after 3 attempts — giving up for now.
[chunk_size=1] 12/189: searching ['recosbynesa']
[chunk_size=1] 13/189: searching ['findingehl22']
[chunk_size=1] 14/189: searching ['jhoanagrasya0614']
[chunk_size=1] 15/189: searching ['

KeyboardInterrupt: 

In [ ]:
still_not_found